# Detecção de Anomalias em Transações (Fraude) — Notebook Google Colab

Este notebook executa o mesmo pipeline do projeto (`main.py`), célula por célula, para quem prefere trabalhar no Google Colab.

**Como usar no Colab:**
1. Faça upload da pasta `src/` inteira (ou do `.zip` do projeto) para o ambiente do Colab.
2. Execute as células em ordem.
3. Para usar seu próprio CSV, defina `CSV_PATH` na célula de configuração.

In [ ]:
# 0. Instalação de dependências (rodar só uma vez no Colab)
!pip install -q imbalanced-learn xgboost lightgbm catboost shap

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())  # garante que a pasta 'src' seja encontrada

from src import data_gen, eda, feature_engineering, balancing, models, evaluation, explainability
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

TARGET_COL = 'Class'
CSV_PATH = None  # ex: 'creditcard.csv' -- deixe None para usar dados sintéticos

## 1. Carregamento dos dados

In [ ]:
if CSV_PATH:
    df_raw = data_gen.load_dataset(csv_path=CSV_PATH)
else:
    df_raw = data_gen.load_dataset(n_samples=30000, fraud_ratio=0.015, random_state=42)

df_raw.head()

## 2. Análise Exploratória dos Dados (EDA)

In [ ]:
eda_results = eda.run_eda(df_raw, target_col=TARGET_COL, output_dir='outputs/eda')
print(eda_results['qualidade_dados'])
eda_results['balanceamento_classes']

In [ ]:
df_clean = eda.clean_basic_issues(df_raw, target_col=TARGET_COL)
df_clean.shape

## 3. Engenharia de Recursos

In [ ]:
df_fe = feature_engineering.engineer_features(df_clean, time_col='Time', amount_col='Amount')
feature_cols = [c for c in df_fe.columns if c != TARGET_COL]
numeric_cols_to_scale = [c for c in feature_cols if df_fe[c].dtype != 'object' and not str(df_fe[c].dtype).startswith('category')]
df_scaled, scaler = feature_engineering.scale_features(df_fe, columns=numeric_cols_to_scale, method='robust')
df_scaled[feature_cols].head()

## 4. Split treino/teste + Balanceamento (SMOTE)

In [ ]:
X = df_scaled[feature_cols]
y = df_scaled[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

X_train_bal, y_train_bal = balancing.balance_data(X_train, y_train, strategy='smote', random_state=42)
balancing.balance_report(y_train, y_train_bal)

## 5. Treinamento e comparação de modelos

In [ ]:
n_neg, n_pos = (y_train_bal == 0).sum(), max(1, (y_train_bal == 1).sum())
scale_pos_weight = n_neg / n_pos

zoo = models.ModelZoo(random_state=42, contamination=float(y.mean()))
supervised_models = models.get_supervised_models(zoo, scale_pos_weight=scale_pos_weight)

results, trained_models = {}, {}
for name, model in supervised_models.items():
    model.fit(X_train_bal, y_train_bal)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results[name] = evaluation.evaluate_predictions(y_test.values, y_pred, y_proba)
    trained_models[name] = model
    print(name, results[name])

In [ ]:
comparison_df = evaluation.compare_models(results)
comparison_df

## 6. Modelos não supervisionados (Isolation Forest / One-Class SVM)

In [ ]:
anomaly_models = models.get_anomaly_models(zoo)
for name, model in anomaly_models.items():
    model.fit(X_train)
    y_pred_anom = models.anomaly_scores_to_labels(model.predict(X_test))
    print(name, evaluation.evaluate_predictions(y_test.values, y_pred_anom))

## 7. Validação cruzada estratificada do melhor modelo

In [ ]:
best_model_name = comparison_df.index[0]
best_model = trained_models[best_model_name]
print('Melhor modelo:', best_model_name)

cv_scores = evaluation.stratified_cross_validation(best_model, X_train_bal, y_train_bal, scoring='average_precision', n_splits=5)
print('PR-AUC por fold:', np.round(cv_scores, 4))
print('PR-AUC médio:', cv_scores.mean().round(4), '+/-', cv_scores.std().round(4))

## 8. Explicabilidade (SHAP)

In [ ]:
shap_values, explain_sample = explainability.compute_shap_values(best_model, X_train_bal, X_test)
explainability.plot_shap_summary(shap_values, 'outputs/shap_importancia_global.png')

from IPython.display import Image
Image('outputs/shap_importancia_global.png')

In [ ]:
preds_on_sample = best_model.predict(explain_sample)
fraud_positions = np.where(preds_on_sample == 1)[0]
explain_idx = int(fraud_positions[0]) if len(fraud_positions) > 0 else 0

explainability.plot_shap_waterfall(shap_values, explain_idx, 'outputs/shap_transacao_individual.png')
explainability.top_features_for_transaction(shap_values, explain_idx)